# MovieLens Data Processing Exercise

This beginner-level assignment practices a complete data-processing workflow using only Python's standard library and built-in collections. You will download and extract MovieLens data, read CSV files, write and reload JSON Lines files, aggregate and combine data, filter and sort the results, and save a final CSV file.

**Input files**

- `C:/data/movielens/ml-latest-small/movies.csv`
- `C:/data/movielens/ml-latest-small/ratings.csv`

**Output files**

- `C:/data/output/movielens/movies/movies.json`
- `C:/data/output/movielens/ratings/ratings.json`
- `C:/data/output/movielens/popular_movies/popular_movies.csv`


## Learning objectives

Learn to read and write CSV and JSON data with the standard library and use lists, dictionaries, sets, and `defaultdict` to aggregate, combine, filter, and sort records.


## 1. Import required libraries

In [ ]:
from pathlib import Path
from collections import defaultdict
import csv
import json
import urllib.request
import zipfile


## 2. Define paths and create output directories

The path variables below keep file locations in one place. Complete the TODO using `Path.mkdir()` with options that create missing parent directories and do not fail when a directory already exists.

In [ ]:
DATASET_DIR = Path("C:/data/movielens/ml-latest-small")
MOVIES_CSV = DATASET_DIR / "movies.csv"
RATINGS_CSV = DATASET_DIR / "ratings.csv"

OUTPUT_BASE_DIR = Path("C:/data/output/movielens")
MOVIES_JSON_DIR = OUTPUT_BASE_DIR / "movies"
RATINGS_JSON_DIR = OUTPUT_BASE_DIR / "ratings"
POPULAR_MOVIES_DIR = OUTPUT_BASE_DIR / "popular_movies"

MOVIES_JSON_FILE = MOVIES_JSON_DIR / "movies.json"
RATINGS_JSON_FILE = RATINGS_JSON_DIR / "ratings.json"
POPULAR_MOVIES_CSV = POPULAR_MOVIES_DIR / "popular_movies.csv"

# Safely create all output directories.
for output_dir in (MOVIES_JSON_DIR, RATINGS_JSON_DIR, POPULAR_MOVIES_DIR):
    output_dir.mkdir(parents=True, exist_ok=True)


## 3. Download and extract the dataset

Dataset ZIP: https://files.grouplens.org/datasets/movielens/ml-latest-small.zip  
Dataset information: https://grouplens.org/datasets/movielens/

1. Download `ml-latest-small.zip`.
2. Save it under `C:/data/movielens`.
3. Extract the ZIP file there.
4. Confirm that `movies.csv` and `ratings.csv` are in `C:/data/movielens/ml-latest-small`.

You may do this manually or complete the optional starter code below.

In [ ]:
# Optional download/extraction starter code
DATASET_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
DOWNLOAD_DIR = Path("C:/data/movielens")
ZIP_FILE = DOWNLOAD_DIR / "ml-latest-small.zip"

DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)

if not MOVIES_CSV.exists() or not RATINGS_CSV.exists():
    urllib.request.urlretrieve(DATASET_URL, ZIP_FILE)
    with zipfile.ZipFile(ZIP_FILE) as archive:
        archive.extractall(DOWNLOAD_DIR)


In [ ]:
# Validate the required input files.
assert MOVIES_CSV.exists(), f"File not found: {MOVIES_CSV}"
assert RATINGS_CSV.exists(), f"File not found: {RATINGS_CSV}"
print("MovieLens input files are ready.")

## Exercise 1 — Load `movies.csv`

Read the file into a list of dictionaries named `movies`. Display its first five records and print its row and column counts. Expected keys are `movieId`, `title`, and `genres`.


In [ ]:
with MOVIES_CSV.open(encoding="utf-8", newline="") as file:
    movies = [
        {**row, "movieId": int(row["movieId"])}
        for row in csv.DictReader(file)
    ]

print(*movies[:5], sep="\n")
print(f"Rows: {len(movies)}, Columns: {len(movies[0]) if movies else 0}")
print(f"Columns: {list(movies[0]) if movies else []}")


## Exercise 2 — Write movie data as JSON

Write every dictionary in `movies` to `MOVIES_JSON_FILE` as one JSON object per line.


In [ ]:
with MOVIES_JSON_FILE.open("w", encoding="utf-8") as file:
    for movie in movies:
        file.write(json.dumps(movie, ensure_ascii=False) + "\n")


## Exercise 3 — Load `ratings.csv`

Read the file into a list of dictionaries named `ratings`. Convert IDs and timestamps to integers and ratings to floats. Display five records, print its dimensions, and count missing values for every field.


In [ ]:
with RATINGS_CSV.open(encoding="utf-8", newline="") as file:
    ratings = [
        {
            "userId": int(row["userId"]),
            "movieId": int(row["movieId"]),
            "rating": float(row["rating"]),
            "timestamp": int(row["timestamp"]),
        }
        for row in csv.DictReader(file)
    ]

print(*ratings[:5], sep="\n")
print(f"Rows: {len(ratings)}, Columns: {len(ratings[0]) if ratings else 0}")
missing_counts = {
    column: sum(record.get(column) in (None, "") for record in ratings)
    for column in (ratings[0] if ratings else {})
}
print("Missing values:", missing_counts)


## Exercise 4 — Write rating data as JSON

Write every dictionary in `ratings` to `RATINGS_JSON_FILE` as one JSON object per line.


In [ ]:
with RATINGS_JSON_FILE.open("w", encoding="utf-8") as file:
    for rating in ratings:
        file.write(json.dumps(rating) + "\n")


## Exercise 5 — Load the ratings JSON file

Load `RATINGS_JSON_FILE` into a list of dictionaries named `ratings_from_json`, then verify that it has the same number of records as `ratings`.


In [ ]:
with RATINGS_JSON_FILE.open(encoding="utf-8") as file:
    ratings_from_json = [json.loads(line) for line in file if line.strip()]

assert len(ratings_from_json) == len(ratings), "Ratings row counts do not match."


## Exercise 6 — Calculate movie rating statistics

Using `ratings_from_json`, calculate each movie's distinct rater count (`user_count`) and average rating (`avg_rating`). Store dictionaries with these values in `movie_stats`.


In [ ]:
rating_groups = defaultdict(lambda: {"users": set(), "rating_total": 0.0, "rating_count": 0})

for rating in ratings_from_json:
    group = rating_groups[rating["movieId"]]
    group["users"].add(rating["userId"])
    group["rating_total"] += rating["rating"]
    group["rating_count"] += 1

movie_stats = [
    {
        "movieId": movie_id,
        "user_count": len(values["users"]),
        "avg_rating": values["rating_total"] / values["rating_count"],
    }
    for movie_id, values in rating_groups.items()
]


## Exercise 7 — Find popular highly rated movies

Keep movies having at least 100 distinct raters and an average rating of at least 4.0. Store them in `filtered_movies`.


In [ ]:
filtered_movies = [
    stat for stat in movie_stats
    if stat["user_count"] >= 100 and stat["avg_rating"] >= 4.0
]


## Exercise 8 — Add movie title and genres

Use a dictionary keyed by `movieId` to combine `filtered_movies` with `movies`. Store the required fields in `popular_movies_with_details`.


In [ ]:
movies_by_id = {movie["movieId"]: movie for movie in movies}
popular_movies_with_details = [
    {
        "movieId": stat["movieId"],
        "title": movies_by_id[stat["movieId"]]["title"],
        "genres": movies_by_id[stat["movieId"]]["genres"],
        "user_count": stat["user_count"],
        "avg_rating": stat["avg_rating"],
    }
    for stat in filtered_movies
    if stat["movieId"] in movies_by_id
]


## Exercise 9 — Sort the result

Sort by `avg_rating` descending, then by `user_count` descending for ties. Store the result in `popular_movies`.


In [ ]:
popular_movies = sorted(
    popular_movies_with_details,
    key=lambda movie: (movie["avg_rating"], movie["user_count"]),
    reverse=True,
)


## Exercise 10 — Write the result as CSV

Write `popular_movies` to `POPULAR_MOVIES_CSV`, including the header.


In [ ]:
FIELDNAMES = ["movieId", "title", "genres", "user_count", "avg_rating"]
with POPULAR_MOVIES_CSV.open("w", encoding="utf-8", newline="") as file:
    writer = csv.DictWriter(file, fieldnames=FIELDNAMES)
    writer.writeheader()
    writer.writerows(popular_movies)


## Verification

Run these checks only after completing all exercises. They validate the required files and key result rules without showing how to build the result.

In [ ]:
assert movies, "movies must not be empty."
assert ratings, "ratings must not be empty."
assert MOVIES_JSON_FILE.exists(), f"File not found: {MOVIES_JSON_FILE}"
assert RATINGS_JSON_FILE.exists(), f"File not found: {RATINGS_JSON_FILE}"
assert ratings_from_json, "ratings_from_json must not be empty."
assert len(ratings_from_json) == len(ratings), "Ratings row counts do not match."

EXPECTED_FIELDS = ["movieId", "title", "genres", "user_count", "avg_rating"]
assert all(list(movie) == EXPECTED_FIELDS for movie in popular_movies), "Unexpected fields or order."
assert all(movie["user_count"] >= 100 for movie in popular_movies)
assert all(movie["avg_rating"] >= 4.0 for movie in popular_movies)
assert popular_movies == sorted(
    popular_movies,
    key=lambda movie: (movie["avg_rating"], movie["user_count"]),
    reverse=True,
), "The result is not sorted correctly."
assert POPULAR_MOVIES_CSV.exists(), f"File not found: {POPULAR_MOVIES_CSV}"
print("All validation checks passed.")


## Student submission requirements

Submit the completed notebook and the three generated output files. Restart the kernel and run the notebook from top to bottom before submission. Use meaningful names, `pathlib.Path` for paths, standard-library modules for file handling, and Python collections for every transformation.
